# Stage 13B.4A — Historical Land-Cover Fallback Diagnosis & Pre-Event Recovery

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

**Goal:** diagnose the 2016–2021 land-cover failures and test an Earth Engine Dynamic World fallback using **only observations before each historical reference date**. This stage does not change or retrain any model.

**Data safeguards**
- Keep the Stage 13B.4 exported rainfall, soil moisture and waterway values unchanged.
- Use `GOOGLE/DYNAMICWORLD/V1`, label mode over a strictly pre-event window; also count valid label observations at every point. Never fill missing land cover with arbitrary categories.
- Do **not** equate worldwide image count with valid pixel coverage at a point.
- For historical GLC_FCS30D, check whether local data are annual single-band rasters or multi-band 2000–2022 tiles; do not treat a tile-name year-range as an individual annual map.
- Use the original Stage 05B categorical map, reliability threshold, and water-control restriction. A pre-event Dynamic World category is *candidate-valid* only when enough observations exist.
- No final controls are selected at this stage. Spatial validity, uniqueness, group assembly and training authorization remain **pending**.


## PART A — Configuration and original output audit

In [1]:
from pathlib import Path
import json
import re
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
DEV_ROOT = PROJECT_ROOT / "Landslide_FYP_Development"
PREVIOUS_DIR = DEV_ROOT / "05_model_results" / "stage13b4_historical_control_features"
OUTPUT_DIR = DEV_ROOT / "05_model_results" / "stage13b4a_landcover_recovery"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PREVIOUS_DIR / "replacement_candidate_extracted_features.csv"
MANIFEST_PATH = DEV_ROOT / "05_model_results" / "stage13b3_control_recovery" / "candidate_temporal_extraction_manifest.csv"

EE_PROJECT = "fyp-landslide-malaysia"
RUN_EE = True
EARTH_ENGINE_SCALE_M = 10
PRE_EVENT_WINDOW_DAYS = 365
MIN_VALID_DW_OBSERVATIONS = 3  # conservative threshold for a point-level mode
PAUSE_SECONDS = 0.1

# Optional: set this to the existing local historical land-cover directory to audit TIFF structure.
LANDCOVER_ROOT = PROJECT_ROOT / "Landslide_FYP_Data" / "03_Factors" / "Landcover"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Stage 13B.4 output missing: {INPUT_PATH}")

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_DIR)


Input: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b4_historical_control_features\replacement_candidate_extracted_features.csv
Output: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b4a_landcover_recovery


In [2]:
df = pd.read_csv(INPUT_PATH, low_memory=False)
required = {"matched_positive_id", "old_control_sample_id", "sample_id", "longitude", "latitude", "reference_date", "landcover_harmonised", "landcover_status", "full_temporal_row_valid"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["reference_date"] = pd.to_datetime(df["reference_date"], errors="coerce")
if df["reference_date"].isna().any():
    raise ValueError("Invalid historical reference dates; do not impute")
df["reference_year"] = df["reference_date"].dt.year
for c in ["longitude", "latitude"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
if df[["longitude", "latitude"]].isna().any().any():
    raise ValueError("Invalid coordinates present")

print("Total shortlist rows:",len(df))
print("Unique candidate/date pairs:",len(df.drop_duplicates(["sample_id", "reference_date"])))
print("Current strict land-cover valid:",int(df["landcover_valid_strict"].astype(str).str.lower().eq("true").sum()))
print(df.groupby(["reference_year", "landcover_status"], dropna=False).size().to_string())


Total shortlist rows: 170
Unique candidate/date pairs: 154
Current strict land-cover valid: 45
reference_year  landcover_status                    
2008            ok                                       5
2011            ok                                      10
2012            ok                                       5
2013            ok                                      10
2014            ok                                      15
2016            local_year_missing_or_no_valid_pixel    35
2017            local_year_missing_or_no_valid_pixel    40
2018            local_year_missing_or_no_valid_pixel     5
2021            local_year_missing_or_no_valid_pixel    45


## PART B — Check local GLC_FCS30D files and whether years are stored in raster bands

In [3]:
# Diagnostic only; does not sample these products or change prior results.
import rasterio

raster_rows = []
paths = sorted(LANDCOVER_ROOT.rglob("*.tif")) if LANDCOVER_ROOT.exists() else []
for p in paths:
    try:
        with rasterio.open(p) as src:
            years_in_name = re.findall(r"(?:19|20)\d{2}",p.stem)
            raster_rows.append({
                "file":str(p), "name":p.name, "band_count":src.count,
                "years_in_name":";".join(years_in_name),
                "descriptions":str(src.descriptions[:6]), "crs":str(src.crs),
                "nodata":str(src.nodata), "extent":str(tuple(round(float(x),3) for x in src.bounds)),
                "possible_multiyear_stack":bool(src.count>1),
            })
    except Exception as exc:
        raster_rows.append({"file":str(p),"name":p.name,"error":str(exc)[:300]})
local_audit=pd.DataFrame(raster_rows,columns=["file","name","band_count","years_in_name","descriptions","crs","nodata","extent","possible_multiyear_stack","error"])
local_audit.to_csv(OUTPUT_DIR / "local_glc_raster_structure_audit.csv",index=False)
print("Raster files:",len(local_audit))
print("Multiband rasters:",int(local_audit["possible_multiyear_stack"].fillna(False).sum()))
print("WARNING: filenames such as 20002022 indicate a multi-year period, not necessarily a single-year raster")
print(local_audit[["name","band_count","years_in_name","possible_multiyear_stack"]].head(20).to_string(index=False))


Raster files: 16
Multiband rasters: 0
                                      name  band_count years_in_name  possible_multiyear_stack
Malaysia_GLC_FCS30D_2007_East_Malaysia.tif           1          2007                     False
   Malaysia_GLC_FCS30D_2007_Peninsular.tif           1          2007                     False
Malaysia_GLC_FCS30D_2008_East_Malaysia.tif           1          2008                     False
   Malaysia_GLC_FCS30D_2008_Peninsular.tif           1          2008                     False
Malaysia_GLC_FCS30D_2009_East_Malaysia.tif           1          2009                     False
   Malaysia_GLC_FCS30D_2009_Peninsular.tif           1          2009                     False
Malaysia_GLC_FCS30D_2010_East_Malaysia.tif           1          2010                     False
   Malaysia_GLC_FCS30D_2010_Peninsular.tif           1          2010                     False
Malaysia_GLC_FCS30D_2011_East_Malaysia.tif           1          2011                     False
   Malaysia_

## PART C — Dynamic World point-level source verification

**Main recovery design:** for each candidate/reference-date pair, use Dynamic World images from the **365 days before** the reference date (exclusive), calculate the modal label at the point, and check at least three unmasked pixel observations. The original Stage 05B annual source used some full-year composites; those are *not* used for our primary recovery because they may include post-event imagery.

Dynamic World starts on 27 June 2015. For some early events, the pre-event observation window may be partly unavailable. This is explicitly flagged.


In [4]:
if RUN_EE:
    import ee
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception as exc:
        raise RuntimeError("Earth Engine authorization failed. Use ee.Authenticate() in an interactive cell, then rerun. " + str(exc))

unique = df.drop_duplicates(["sample_id","reference_date"]).copy().reset_index(drop=True)
unique["extract_row_id"] = np.arange(len(unique),dtype=int)
unique["reference_date_str"] = unique["reference_date"].dt.strftime("%Y-%m-%d")
needs = unique.loc[unique["landcover_harmonised"].isna()].copy()
print("Unique candidate/date pairs requiring land-cover recovery:",len(needs))
print("By year:",needs["reference_year"].value_counts().sort_index().to_dict())


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Unique candidate/date pairs requiring land-cover recovery: 109
By year: {2016: 30, 2017: 40, 2018: 5, 2021: 34}


In [5]:
def dw_pre_event_for_group(date_str, frame):
    end=ee.Date(date_str)
    start=end.advance(-PRE_EVENT_WINDOW_DAYS,"day")
    collection=ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterDate(start,end)
    collection=collection.filterBounds(ee.Geometry.MultiPoint(frame[["longitude","latitude"]].astype(float).values.tolist()).bounds().buffer(1500))
    scene_count=int(collection.size().getInfo())
    points=ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([float(r.longitude),float(r.latitude)]),{"extract_row_id":int(r.extract_row_id)})
        for r in frame.itertuples(index=False)
    ])
    if scene_count==0:
        return [{"extract_row_id":int(v),"dw_label_pre_event":np.nan,"dw_valid_obs":0,"dw_scene_count_roi":0,"dw_error":"no_scenes"} for v in frame["extract_row_id"]]
    labels=collection.select("label")
    image=ee.Image.cat([
        labels.mode().rename("dw_label_pre_event"),
        labels.count().rename("dw_valid_obs"),
    ])
    data=image.reduceRegions(collection=points,reducer=ee.Reducer.first(),scale=EARTH_ENGINE_SCALE_M).getInfo()
    results=[]
    for row in data.get("features",[]):
        prop=row.get("properties",{})
        results.append({
            "extract_row_id":prop.get("extract_row_id"),
            "dw_label_pre_event":prop.get("dw_label_pre_event"),
            "dw_valid_obs":prop.get("dw_valid_obs"),
            "dw_scene_count_roi":scene_count,
            "dw_error":"",
        })
    return results

run_rows=[]
if RUN_EE:
    for date_str,part in needs.groupby("reference_date_str",sort=True):
        try:
            rows=dw_pre_event_for_group(date_str,part)
            run_rows.extend(rows)
            print(date_str,"points",len(part),"returned",len(rows),"roi scenes",max((r.get("dw_scene_count_roi",0) for r in rows),default=0))
        except Exception as exc:
            message=str(exc)[:450]
            print("ERROR",date_str,message)
            run_rows.extend({"extract_row_id":int(v),"dw_label_pre_event":np.nan,"dw_valid_obs":np.nan,"dw_scene_count_roi":np.nan,"dw_error":message} for v in part["extract_row_id"])
        time.sleep(PAUSE_SECONDS)

dw=pd.DataFrame(run_rows,columns=["extract_row_id","dw_label_pre_event","dw_valid_obs","dw_scene_count_roi","dw_error"])
if len(dw) and dw["extract_row_id"].duplicated().any():
    raise ValueError("Duplicate extracted row ID")
if RUN_EE and len(dw)!=len(needs):
    raise ValueError("Not all candidate/date pairs were returned from EE")
dw.to_csv(OUTPUT_DIR / "dynamic_world_pre_event_point_extraction.csv",index=False)
print("Extracted unique point/dates:",len(dw))


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


2016-07-01 points 5 returned 5 roi scenes 8


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


2016-07-22 points 5 returned 5 roi scenes 18


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


...

2021-12-12 points 5 returned 5 roi scenes 34


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


2021-12-18 points 8 returned 8 roi scenes 34


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


2021-12-19 points 6 returned 6 roi scenes 34
Extracted unique point/dates: 109


## PART D — Harmonize recovered classes without changing original QC

In [6]:
DW_MAP={0:"WATER",1:"FOREST",2:"GRASS",3:"WETLAND",4:"CROPLAND",5:"SHRUB",6:"BUILT",7:"BARE",8:"OTHER"}

unique=unique.merge(dw,on="extract_row_id",how="left",validate="one_to_one")
unique["dw_valid_obs"] = pd.to_numeric(unique["dw_valid_obs"],errors="coerce")
unique["dw_label_pre_event"] = pd.to_numeric(unique["dw_label_pre_event"],errors="coerce")
unique["dw_recovered_class"] = unique["dw_label_pre_event"].map(DW_MAP)
unique["dw_pre_event_adequate"] = (
    unique["dw_recovered_class"].notna()
    & unique["dw_valid_obs"].ge(MIN_VALID_DW_OBSERVATIONS)
)
unique["dw_pre_event_control_valid"] = (
    unique["dw_pre_event_adequate"]
    & unique["dw_recovered_class"].ne("WATER")
)
unique["landcover_recovered_candidate"] = (
    unique["landcover_harmonised"].isna()
    & unique["dw_pre_event_control_valid"]
)
unique["revised_landcover_class_candidate"] = unique["landcover_harmonised"].astype("object")
mask=unique["landcover_recovered_candidate"]
if mask.any():
    unique.loc[mask,"revised_landcover_class_candidate"]=unique.loc[mask,"dw_recovered_class"].values
unique["revised_landcover_source_candidate"] = unique["landcover_source"].astype("object")
if mask.any():
    unique.loc[mask,"revised_landcover_source_candidate"]="Dynamic_World_PRE_EVENT_365D"
unique["recovery_status"] = np.select(
    [unique["landcover_harmonised"].notna(),unique["landcover_recovered_candidate"]],
    ["PREVIOUSLY_VALID_OR_CLASSIFIED","PRE_EVENT_DW_CANDIDATE_PASS"],
    default="UNRECOVERED_REVIEW_OBSERVATIONS",
)
unique.to_csv(OUTPUT_DIR / "landcover_recovery_unique_candidate_date.csv",index=False)
print(unique["recovery_status"].value_counts().to_string())


recovery_status
PRE_EVENT_DW_CANDIDATE_PASS        74
PREVIOUSLY_VALID_OR_CLASSIFIED     45
UNRECOVERED_REVIEW_OBSERVATIONS    35


## PART E — Slot coverage, recovery decision and audit exports

This section joins only by **candidate ID + reference date**, preserves earlier rainfall/soil-moisture/waterway QC, and reports whether each replacement slot has at least one *feature-valid* option. It does not authorize actual replacements.


In [7]:
add_cols=["sample_id","reference_date","dw_label_pre_event","dw_valid_obs","dw_scene_count_roi","dw_error", "dw_recovered_class","dw_pre_event_adequate","dw_pre_event_control_valid","landcover_recovered_candidate","revised_landcover_class_candidate","revised_landcover_source_candidate","recovery_status"]
merged=df.merge(unique[add_cols],on=["sample_id","reference_date"],how="left",validate="many_to_one")
for col in ["rainfall_valid","soil_moisture_complete","waterway_available"]:
    merged[col+"_bool"] = merged[col].astype(str).str.lower().eq("true")
merged["pre_event_feature_valid_candidate"] = (
    merged["rainfall_valid_bool"]
    & merged["soil_moisture_complete_bool"]
    & merged["waterway_available_bool"]
    & merged["revised_landcover_class_candidate"].notna()
    & merged["revised_landcover_class_candidate"].ne("WATER")
    & (merged["landcover_valid_strict"].astype(str).str.lower().eq("true") | merged["landcover_recovered_candidate"].fillna(False))
)
merged["verified_replacement"] = False
merged["spatial_exclusion_verified"] = False
merged["control_assignment_verified"] = False
merged.to_csv(OUTPUT_DIR / "replacement_candidates_after_landcover_recovery.csv",index=False)
slot_cols=["matched_positive_id","old_control_sample_id"]
slots=merged.groupby(slot_cols,dropna=False).agg(
    available_candidates=("sample_id","size"),
    feature_valid_candidates=("pre_event_feature_valid_candidate","sum"),
    dw_recovery_candidates=("landcover_recovered_candidate","sum"),
).reset_index()
slots["slot_has_options"]=slots["feature_valid_candidates"].gt(0)
slots.to_csv(OUTPUT_DIR / "replacement_slot_readiness_after_landcover_recovery.csv",index=False)
groups=slots.groupby("matched_positive_id").agg(
    slots_required=("old_control_sample_id","size"),
    slots_with_candidates=("slot_has_options","sum"),
    feature_valid_candidate_options=("feature_valid_candidates","sum"),
).reset_index()
groups["all_slots_have_candidates"] = groups["slots_required"].eq(groups["slots_with_candidates"])
groups.to_csv(OUTPUT_DIR / "group_readiness_after_landcover_recovery.csv",index=False)
print("Slots with provisional feature options:",int(slots["slot_has_options"].sum()),"/",len(slots))
print("Groups with provisional feature options for all slots:",int(groups["all_slots_have_candidates"].sum()),"/",len(groups))


Slots with provisional feature options: 33 / 34
Groups with provisional feature options for all slots: 32 / 33


In [8]:
summary={
    "input_shortlist_rows":int(len(df)),
    "unique_candidate_date_pairs":int(len(unique)),
    "original_missing_landcover_unique_pairs":int(needs.shape[0]),
    "new_pre_event_dw_adequate_nonwater_pairs":int(unique["landcover_recovered_candidate"].sum()),
    "new_feature_valid_rows_after_recovery":int(merged["pre_event_feature_valid_candidate"].sum()),
    "slots_with_provisional_options":int(slots["slot_has_options"].sum()),
    "groups_with_provisional_options_for_all_slots":int(groups["all_slots_have_candidates"].sum()),
    "total_target_groups":int(len(groups)),
    "earth_engine_failed_rows":int(unique["dw_error"].fillna("").ne("").sum()) if "dw_error" in unique.columns else 0,
    "verified_recovered_groups":0,
    "retraining_authorized":False,
    "temporal_policy":"pre_event_365d_strictly_before_event_date; >=3 valid observations; water controls excluded",
    "caveat":"Annual historical land cover retained for preexisting valid rows may include post-event observations; assess before final scientific claims",
}
(OUTPUT_DIR / "stage13b4a_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
gate=pd.DataFrame([
    {"check":"Local TIFF / multiband structure audited","status":"DONE","details":str(len(local_audit))},
    {"check":"Pre-event Dynamic World point-level check","status":"DONE" if RUN_EE else "NOT_RUN","details":str(len(dw))},
    {"check":"Recovered controls meet date / feature QC","status":"PROVISIONAL_ONLY","details":str(summary["new_pre_event_dw_adequate_nonwater_pairs"])},
    {"check":"Original pseudoabsence spatial exclusion","status":"PENDING","details":"Not tested in this stage"},
    {"check":"Nonreuse & 1:2 group assignment","status":"PENDING","details":"Not tested in this stage"},
    {"check":"Temporal leakage / reference year policy","status":"REVIEW_REQUIRED","details":"Existing annual layers may include post-event observations"},
    {"check":"Dataset rebuild / three-model retraining","status":"NOT_AUTHORIZED","details":"Run Stage 13B.5 first"},
])
gate.to_csv(OUTPUT_DIR / "stage13b4a_decision_gate.csv",index=False)
print(json.dumps(summary,indent=2))


{
  "input_shortlist_rows": 170,
  "unique_candidate_date_pairs": 154,
  "original_missing_landcover_unique_pairs": 109,
  "new_pre_event_dw_adequate_nonwater_pairs": 74,
  "new_feature_valid_rows_after_recovery": 132,
  "slots_with_provisional_options": 33,
  "groups_with_provisional_options_for_all_slots": 32,
  "total_target_groups": 33,
  "earth_engine_failed_rows": 0,
  "verified_recovered_groups": 0,
  "retraining_authorized": false,
  "temporal_policy": "pre_event_365d_strictly_before_event_date; >=3 valid observations; water controls excluded",
  "caveat": "Annual historical land cover retained for preexisting valid rows may include post-event observations; assess before final scientific claims"
}


### Send these outputs for review

1. `stage13b4a_summary.json`
2. `stage13b4a_decision_gate.csv`
3. `local_glc_raster_structure_audit.csv`
4. `dynamic_world_pre_event_point_extraction.csv`
5. `replacement_slot_readiness_after_landcover_recovery.csv`
6. `group_readiness_after_landcover_recovery.csv`
7. `replacement_candidates_after_landcover_recovery.csv`

**Next stage:** Stage 13B.5 would check candidate uniqueness, spatial exclusion buffers, accepted 1:2 group formation and temporal safeguards. This notebook deliberately outputs *zero final recovered groups*. If pre-event Dynamic World still produces no viable pixels, obtain the relevant **GLC_FCS30D 2000–2022 multi-band tiles** for the Malaysia locations and use documented band-to-year mapping; do not interpret years embedded in a filename as separate annual rasters.
